### Fase 1: Análise Exploratória de Dados (EDA)

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

df = pd.read_csv("/Workspace/Users/leonardobadell@protonmail.com/CodigosDeAula-Python-Dados-/Projeto/E Commerce Dataset.xlsx - E Comm.csv")
print("E Commerce Dataset carregado com sucesso!\n")


# Primeiras linhas do dataset
print("Visualização das primeiras linhas do dataset\n")
display(df.head(10))

# Remover coluna CustomerID (não é relevante para análise)
df = df.drop(columns=['CustomerID'])
print("\nColuna 'CustomerID' removida do dataset (sem valor preditivo)\n")


# Valores estatísticos totais do dataset
print("Cálculo estatístico total:\n")
display(df.describe())


# Tipo de dados por coluna
print("Tipo de dados por coluna do dataset:\n")
df.info()

print("=" * 50)
print("Verificando se há valores nulos:")
print("=" * 50)
display(df.isnull().sum())


In [0]:
# ==================== ANÁLISE DE DESBALANCEAMENTO DOS DADOS ====================
print("="*70)
print("ANÁLISE EXPLORATÓria DE DADOS - CHURN")
print("="*70)

# Contando quantos clientes fizeram churn e quantos não fizeram
churn_counts = df['Churn'].value_counts()
total_clientes = len(df)

print(f"\nTotal de registros: {total_clientes}")
print(f"Classe 0 (Não Churn): {churn_counts[0]} ({churn_counts[0]/total_clientes*100:.2f}%)")
print(f"Classe 1 (Churn): {churn_counts[1]} ({churn_counts[1]/total_clientes*100:.2f}%)")
print(f"Razão de desbalanceamento: {churn_counts[0]/churn_counts[1]:.2f}:1")
print("\nCONCLUSÃO: Dataset está DESBALANCEADO!")
print("="*70)

# ==================== GRÁFICO 1: BARRAS COM TOTAIS ====================
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(['Não Churn (0)', 'Churn (1)'], 
              [churn_counts[0], churn_counts[1]], 
              color=['green', 'red'], 
              edgecolor='black')

ax.set_ylabel('Quantidade de Clientes', fontsize=12)
ax.set_title('Gráfico 1: Distribuição de Classes - Churn', fontsize=14, fontweight='bold')

# Adicionando valores totais nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# ==================== GRÁFICO 2: BOXPLOT ====================
print("\n" + "="*70)
print("GRÁFICO 2: BOXPLOT - Distribuição de Variáveis por Churn")
print("="*70)

# Variáveis para boxplot
variaveis_boxplot = ['Tenure', 'SatisfactionScore', 'DaySinceLastOrder', 
                     'WarehouseToHome', 'CashbackAmount']

# Criar figura com 2 linhas x 3 colunas
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Cores personalizadas: verde para Retido (0), vermelho para Churn (1)
cores = ['#2ecc71', '#e74c3c']

# Criar boxplots para as 5 primeiras variáveis
for i, var in enumerate(variaveis_boxplot):
    # Calcular medianas para cada grupo
    med0 = df[df['Churn']==0][var].median()
    med1 = df[df['Churn']==1][var].median()
    
    # Criar boxplot
    bp = axes[i].boxplot([df[df['Churn']==0][var].dropna(), 
                          df[df['Churn']==1][var].dropna()],
                         tick_labels=['Retido (0)', 'Churn (1)'],
                         patch_artist=True,
                         widths=0.6)
    
    # Aplicar cores
    for patch, color in zip(bp['boxes'], cores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Configurar título e labels
    axes[i].set_title(f'Distribuição: {var} vs Churn', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Status do Cliente', fontsize=10)
    axes[i].set_ylabel(var, fontsize=10)
    
    # Adicionar texto com medianas
    axes[i].text(0.5, 0.98, f'Med(0)={med0:.1f} | Med(1)={med1:.1f}', 
                transform=axes[i].transAxes, fontsize=9,
                verticalalignment='top', horizontalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# GRÁFICO 6: Distribuição de Reclamações vs Churn (gráfico de barras)
reclamacao_churn = pd.crosstab(df['Complain'], df['Churn'])

# Posições das barras
x_pos = [0, 1]
width = 0.35

# Criar gráfico de barras agrupadas
axes[5].bar([p - width/2 for p in x_pos], 
            reclamacao_churn[0], 
            width, 
            label='Retido (0)', 
            color='#2ecc71', 
            edgecolor='black')

axes[5].bar([p + width/2 for p in x_pos], 
            reclamacao_churn[1], 
            width, 
            label='Churn (1)', 
            color='#e74c3c', 
            edgecolor='black')

# Adicionar valores nas barras
for i, (val0, val1) in enumerate(zip(reclamacao_churn[0], reclamacao_churn[1])):
    axes[5].text(i - width/2, val0 + 50, str(val0), 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[5].text(i + width/2, val1 + 20, str(val1), 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[5].set_xlabel('Reclamação', fontsize=10)
axes[5].set_ylabel('Quantidade de Clientes', fontsize=10)
axes[5].set_title('Distribuição: Reclamações vs Churn', fontsize=11, fontweight='bold')
axes[5].set_xticks(x_pos)
axes[5].set_xticklabels(['Sem Reclamação', 'Com Reclamação'])
axes[5].legend()

plt.suptitle('ANÁLISE EXPLORATÓRIA: Variáveis-Chave vs Churn\nE-Commerce Dataset', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# ==================== GRÁFICO 3: HEATMAP DE CORRELAÇÃO ====================
print("\n" + "="*70)
print("GRÁFICO 3: HEATMAP - Correlação de Pearson")
print("="*70)

# Selecionar colunas numéricas relevantes
colunas_heatmap = [
    'Churn',
    'Tenure',
    'Complain',
    'SatisfactionScore',
    'DaySinceLastOrder',
    'OrderCount',
    'CouponUsed',
    'WarehouseToHome',
    'HourSpendOnApp',
    'CashbackAmount',
    'OrderAmountHikeFromlastYear'
]

# Calcular matriz de correlação
correlacao = df[colunas_heatmap].corr()

# Criar heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlacao, 
            annot=True, 
            fmt='.2f', 
            cmap='RdYlBu_r', 
            center=0,
            square=True,
            linewidths=0.5)
plt.title('Correlação de Pearson - Fatores que Afetam Churn', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("Correlações com Churn (ordenadas):")
print("="*70)
churn_corr = correlacao['Churn'].sort_values(ascending=False)
print(churn_corr)
print("="*70)

%md
## 📊 Conclusões da Análise Exploratória de Dados (EDA)

### 🔢 Desbalanceamento de Classes
O dataset apresenta **forte desbalanceamento** com razão de **4.94:1**:
* **Classe 0 (Não Churn):** 4.682 clientes (83.16%)
* **Classe 1 (Churn):** 948 clientes (16.84%)

Este desbalanceamento deve ser considerado durante o treinamento de modelos de machine learning (técnicas como SMOTE, undersampling ou pesos de classe podem ser necessárias).

---

### ❌ Valores Nulos
O dataset possui valores ausentes em diversas colunas:
* **Tenure:** 264 valores nulos
* **WarehouseToHome:** 251 valores nulos
* **HourSpendOnApp:** 255 valores nulos
* **DaySinceLastOrder:** 307 valores nulos
* Outras variáveis também apresentam missings

**Decisão:** A coluna `CustomerID` foi removida por não ter valor preditivo (apenas identificador único).

---

### 📦 Insights dos Boxplots
Os boxplots revelam diferenças claras entre clientes que fizeram churn vs. clientes retidos:

* **Tenure:** Clientes com churn têm tenure **significativamente menor** (permaneceram menos tempo)
* **CashbackAmount:** Clientes que fazem churn receberam **menos cashback**
* **DaySinceLastOrder:** Clientes com churn têm medianas diferentes, sugerindo padrões de compra distintos
* **Reclamações:** Clientes com reclamações apresentam **muito mais churn** (visível no gráfico de barras)

---

### 🔥 Insights do Heatmap de Correlação de Pearson
A análise de correlação identificou as **variáveis mais relevantes** para prever churn:

#### ⭐ **Correlações FORTES:**
* **Tenure (-0.35):** Quanto **MAIS tempo** como cliente, **MENOR o risco** de churn → *Foque em reter clientes nos primeiros meses!*

#### 📊 **Correlações MODERADAS:**
* **Complain (+0.25):** Reclamações **aumentam muito** o risco de churn → *Sistema de atendimento prioritário é essencial*
* **DaySinceLastOrder (-0.16):** Padrões de compra recente influenciam churn
* **CashbackAmount (-0.15):** Cashback funciona como "cola" → *Programas de benefícios reduzem churn*
* **SatisfactionScore (+0.11):** Surpreendentemente, maior satisfação tem leve correlação positiva com churn (possível paradoxo: clientes satisfeitos podem estar dando nota alta antes de sair)

#### ❌ **Variáveis IRRELEVANTES (correlação ~0):**
* HourSpendOnApp, CouponUsed, OrderAmountHikeFromlastYear, OrderCount
* Estas variáveis **não influenciam** significativamente o churn

---

### 🎯 Recomendações de Negócio
1. **Retenção Precoce:** Investir em clientes novos (tenure baixo = alto risco)
2. **Gestão de Reclamações:** Resolver reclamações rapidamente reduz churn drasticamente
3. **Programas de Fidelidade:** Aumentar cashback e benefícios para clientes em risco
4. **Não confiar apenas em pesquisas de satisfação:** A correlação positiva com churn sugere que satisfação declarada nem sempre previne saída